# METER NYU Depth v2 Kaggle Training

Concise training notebook for the METER paper reproduction. Depth is converted from meters to centimeters; depth `<= 1 cm` is masked from training loss and metrics. Most logic lives in importable `.py` files so the same code can be zipped and uploaded with Kaggle Model weights.

In [ ]:
# Optional on Kaggle if packages are missing.
# !pip install -q wandb h5py

In [ ]:
from __future__ import annotations

import gc
import json
import random
import sys
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

In [ ]:
@dataclass(frozen=True)
class NotebookConfig:
    kaggle_dataset_slug: str = "your-nyu-depth-v2-slug"
    local_data_root: str = "../nyu_depth_v2"
    local_code_root: str = ".."
    arch_type: str = "xs"
    input_height: int = 192
    input_width: int = 256
    max_depth_cm: float = 1000.0
    invalid_depth_threshold_cm: float = 1.0
    normalize_rgb: bool = True
    resize_inputs: bool = True
    num_epochs: int = 60
    batch_size: int = 128
    num_workers: int = 4
    prefetch_factor: int = 4
    pin_memory: bool = True
    persistent_workers: bool = True
    num_gpus: int = 1
    pretrained_encoder_path: str | None = None
    pretrained_model_path: str | None = None
    dropout_start: float = 0.0
    dropout: float = 0.1
    attention_dropout_start: float = 0.0
    attention_dropout: float = 0.1
    drop_path_start: float = 0.0
    drop_path: float = 0.1
    regularization_start_epoch: int = 31
    regularization_ramp_epochs: int = 10
    regularization_ramp_schedule: str = "linear"
    learning_rate: float = 1e-3
    encoder_learning_rate: float = 1e-4
    decoder_learning_rate: float = 1e-3
    adam_beta1: float = 0.9
    adam_beta2: float = 0.999
    weight_decay: float = 0.03
    scheduler_type: str = "plateau"  # "step", "cosine", or "plateau"
    scheduler_step_size: int = 20
    scheduler_gamma: float = 0.1
    scheduler_eta_min: float = 1e-6
    scheduler_factor: float = 0.5
    scheduler_patience: int = 4
    scheduler_min_lr: float = 1e-6
    checkpoint_metric: str = "val_rmse_m"
    checkpoint_mode: str = "min"
    use_amp: bool = True
    seed: int = 42
    lambda_1: float = 0.5
    lambda_2: float = 100.0
    lambda_3: float = 100.0
    flip: float = 0.5
    mirror: float = 0.5
    c_swap: float = 0.5
    random_crop: float = 0.5
    shifting_strategy: float = 0.5
    color_low: float = 0.9
    color_high: float = 1.1
    depth_shift_min_cm: float = -10.0
    depth_shift_max_cm: float = 10.0
    color_jitter_start: float = 0.0
    color_jitter: float = 0.5
    color_jitter_brightness: float = 0.2
    color_jitter_contrast: float = 0.2
    color_jitter_saturation: float = 0.2
    gaussian_blur_start: float = 0.0
    gaussian_blur: float = 0.15
    gaussian_blur_kernel: int = 3
    gaussian_blur_sigma_min: float = 0.1
    gaussian_blur_sigma_max: float = 1.5
    random_grayscale_start: float = 0.0
    random_grayscale: float = 0.05
    scheduled_augmentation_start_epoch: int = 31
    scheduled_augmentation_ramp_epochs: int = 10
    scheduled_augmentation_ramp_schedule: str = "linear"
    max_train_samples: int | None = None
    max_val_samples: int | None = None
    output_dir: str = "outputs"
    use_wandb: bool = True
    wandb_project: str = "meter-nyu-depth"
    wandb_run_name: str = "meter-nyu-xs-cm"
    wandb_mode: str = "online"
    vis_every_epochs: int = 10
    vis_num_samples: int = 4
    use_tqdm: bool = False
    log_every_steps: int = 25
    wandb_log_every_steps: int = 25
    checkpoint_name: str = "meter_nyu_best.pt"
    final_checkpoint_name: str = "meter_nyu_final.pt"
    freeze_encoder_epochs: int = 5
    early_stopping_patience: int = 10
    early_stopping_min_delta: float = 0.0
    ema_decay: float = 0.999

CFG = NotebookConfig()

In [ ]:
def running_on_kaggle() -> bool:
    return Path("/kaggle/input").exists()

def resolve_code_root() -> Path:
    candidates = []
    if running_on_kaggle():
        base = Path("/kaggle/input") / CFG.kaggle_dataset_slug
        candidates.extend([base, base / "METER", Path("/kaggle/working")])
    candidates.append(Path(CFG.local_code_root).resolve())
    for path in candidates:
        if (path / "architecture.py").exists():
            return path
    raise FileNotFoundError(f"Could not find architecture.py in {candidates}")

CODE_ROOT = resolve_code_root()
sys.path.insert(0, str(CODE_ROOT))
print("CODE_ROOT =", CODE_ROOT)

In [ ]:
from network import build_METER_model
from data import DataConfig, NYUH5DepthDataset, discover_h5_files, resolve_data_root
from train import BalancedMETERLoss, LossConfig
from train import TrainConfig, build_optimizer, build_scheduler, fit
from utils import seed_everything

In [ ]:
seed_everything(CFG.seed)
random.seed(CFG.seed)
np.random.seed(CFG.seed)
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_root = resolve_data_root(CFG.kaggle_dataset_slug, CFG.local_data_root)
print("DATA_ROOT =", data_root)
print("DEVICE =", device)
print(json.dumps(asdict(CFG), indent=2))

In [ ]:
data_config = DataConfig(
    input_height=CFG.input_height,
    input_width=CFG.input_width,
    max_depth_cm=CFG.max_depth_cm,
    invalid_depth_threshold_cm=CFG.invalid_depth_threshold_cm,
    normalize_rgb=CFG.normalize_rgb,
    resize_inputs=CFG.resize_inputs,
    flip=CFG.flip,
    mirror=CFG.mirror,
    c_swap=CFG.c_swap,
    random_crop=CFG.random_crop,
    shifting_strategy=CFG.shifting_strategy,
    color_low=CFG.color_low,
    color_high=CFG.color_high,
    depth_shift_min_cm=CFG.depth_shift_min_cm,
    depth_shift_max_cm=CFG.depth_shift_max_cm,
    color_jitter_start=CFG.color_jitter_start,
    color_jitter=CFG.color_jitter,
    color_jitter_brightness=CFG.color_jitter_brightness,
    color_jitter_contrast=CFG.color_jitter_contrast,
    color_jitter_saturation=CFG.color_jitter_saturation,
    gaussian_blur_start=CFG.gaussian_blur_start,
    gaussian_blur=CFG.gaussian_blur,
    gaussian_blur_kernel=CFG.gaussian_blur_kernel,
    gaussian_blur_sigma_min=CFG.gaussian_blur_sigma_min,
    gaussian_blur_sigma_max=CFG.gaussian_blur_sigma_max,
    random_grayscale_start=CFG.random_grayscale_start,
    random_grayscale=CFG.random_grayscale,
    scheduled_augmentation_start_epoch=CFG.scheduled_augmentation_start_epoch,
    scheduled_augmentation_ramp_epochs=CFG.scheduled_augmentation_ramp_epochs,
    scheduled_augmentation_ramp_schedule=CFG.scheduled_augmentation_ramp_schedule,
)
train_files = discover_h5_files(data_root, "train", CFG.max_train_samples)
val_files = discover_h5_files(data_root, "val", CFG.max_val_samples)
print(f"train={len(train_files):,}, val={len(val_files):,}")

train_ds = NYUH5DepthDataset(train_files, data_config, train=True)
val_ds = NYUH5DepthDataset(val_files, data_config, train=False)
loader_kwargs = {
    "batch_size": CFG.batch_size,
    "num_workers": CFG.num_workers,
    "pin_memory": CFG.pin_memory and device.type == "cuda",
    "persistent_workers": CFG.persistent_workers and CFG.num_workers > 0,
}
if CFG.num_workers > 0:
    loader_kwargs["prefetch_factor"] = CFG.prefetch_factor
train_loader = DataLoader(train_ds, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_ds, shuffle=False, **loader_kwargs)
with torch.no_grad():
    sample_batch = next(iter(train_loader))
    print(
        sample_batch["image"].shape,
        sample_batch["depth"].shape,
        sample_batch["mask"].shape,
        "valid:",
        int(sample_batch["mask"].sum()),
    )
del sample_batch
gc.collect()

In [ ]:
def _checkpoint_state_dict(path: str) -> dict[str, torch.Tensor]:
    checkpoint = torch.load(path, map_location="cpu")
    if isinstance(checkpoint, dict) and "model" in checkpoint:
        state_dict = checkpoint["model"]
    elif isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    else:
        state_dict = checkpoint
    return {key.replace("module.", "", 1) if key.startswith("module.") else key: value for key, value in state_dict.items()}


def load_pretrained_encoder(model: torch.nn.Module, path: str) -> None:
    state_dict = _checkpoint_state_dict(path)
    encoder_state = {
        key.removeprefix("encoder."): value
        for key, value in state_dict.items()
        if key.startswith("encoder.")
    }
    missing, unexpected = model.encoder.load_state_dict(encoder_state, strict=False)
    print(f"Loaded encoder weights from {path}: missing={len(missing)}, unexpected={len(unexpected)}")


def load_pretrained_model(model: torch.nn.Module, path: str) -> None:
    state_dict = _checkpoint_state_dict(path)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print(f"Loaded full model weights from {path}: missing={len(missing)}, unexpected={len(unexpected)}")


model = build_METER_model(
    device=str(device),
    arch_type=CFG.arch_type,
    dropout=CFG.dropout,
    attention_dropout=CFG.attention_dropout,
    drop_path=CFG.drop_path,
).to(device)
if CFG.pretrained_model_path is not None:
    load_pretrained_model(model, CFG.pretrained_model_path)
elif CFG.pretrained_encoder_path is not None:
    load_pretrained_encoder(model, CFG.pretrained_encoder_path)

num_gpus = min(CFG.num_gpus, torch.cuda.device_count()) if device.type == "cuda" else 0
if num_gpus > 1:
    model = torch.nn.DataParallel(model, device_ids=list(range(num_gpus)))
    print(f"Using {num_gpus} GPUs via DataParallel")
elif device.type == "cuda":
    print(f"Using {num_gpus} GPU(s) — no DataParallel needed")
else:
    print("Using CPU")
criterion = BalancedMETERLoss(LossConfig(
    max_depth_cm=CFG.max_depth_cm,
    invalid_depth_threshold_cm=CFG.invalid_depth_threshold_cm,
    lambda_1=CFG.lambda_1,
    lambda_2=CFG.lambda_2,
    lambda_3=CFG.lambda_3,
)).to(device)
train_config = TrainConfig(
    num_epochs=CFG.num_epochs,
    learning_rate=CFG.learning_rate,
    encoder_learning_rate=CFG.encoder_learning_rate,
    decoder_learning_rate=CFG.decoder_learning_rate,
    adam_beta1=CFG.adam_beta1,
    adam_beta2=CFG.adam_beta2,
    weight_decay=CFG.weight_decay,
    dropout_start=CFG.dropout_start,
    dropout=CFG.dropout,
    attention_dropout_start=CFG.attention_dropout_start,
    attention_dropout=CFG.attention_dropout,
    drop_path_start=CFG.drop_path_start,
    drop_path=CFG.drop_path,
    regularization_start_epoch=CFG.regularization_start_epoch,
    regularization_ramp_epochs=CFG.regularization_ramp_epochs,
    regularization_ramp_schedule=CFG.regularization_ramp_schedule,
    scheduled_augmentation_start_epoch=CFG.scheduled_augmentation_start_epoch,
    scheduled_augmentation_ramp_epochs=CFG.scheduled_augmentation_ramp_epochs,
    scheduled_augmentation_ramp_schedule=CFG.scheduled_augmentation_ramp_schedule,
    scheduler_type=CFG.scheduler_type,
    scheduler_step_size=CFG.scheduler_step_size,
    scheduler_gamma=CFG.scheduler_gamma,
    scheduler_eta_min=CFG.scheduler_eta_min,
    scheduler_factor=CFG.scheduler_factor,
    scheduler_patience=CFG.scheduler_patience,
    scheduler_min_lr=CFG.scheduler_min_lr,
    checkpoint_metric=CFG.checkpoint_metric,
    checkpoint_mode=CFG.checkpoint_mode,
    use_amp=CFG.use_amp,
    max_depth_cm=CFG.max_depth_cm,
    invalid_depth_threshold_cm=CFG.invalid_depth_threshold_cm,
    output_dir=CFG.output_dir,
    checkpoint_name=CFG.checkpoint_name,
    final_checkpoint_name=CFG.final_checkpoint_name,
    use_wandb=CFG.use_wandb,
    wandb_project=CFG.wandb_project,
    wandb_run_name=CFG.wandb_run_name,
    wandb_mode=CFG.wandb_mode,
    vis_every_epochs=CFG.vis_every_epochs,
    vis_num_samples=CFG.vis_num_samples,
    use_tqdm=CFG.use_tqdm,
    log_every_steps=CFG.log_every_steps,
    wandb_log_every_steps=CFG.wandb_log_every_steps,
    freeze_encoder_epochs=CFG.freeze_encoder_epochs,
    early_stopping_patience=CFG.early_stopping_patience,
    early_stopping_min_delta=CFG.early_stopping_min_delta,
    ema_decay=CFG.ema_decay,
)
optimizer = build_optimizer(model, train_config)
scheduler = build_scheduler(optimizer, train_config)
with torch.no_grad():
    dummy = torch.zeros(1, 3, CFG.input_height, CFG.input_width, device=device)
    print("dummy output:", tuple(model(dummy).shape))

In [ ]:
history = fit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    train_config=train_config,
    experiment_config=asdict(CFG),
)

## Package for Kaggle Model

Run this from the repository root after training to create a local zip you can upload manually to Kaggle Model:

```bash
conda run -n mlx python scripts/package_kaggle_model.py   --checkpoint-dir outputs/checkpoints   --output-zip outputs/kaggle_meter_model.zip   --arch-type s
```

Optional API upload, only if your Kaggle credentials are configured:

```bash
conda run -n mlx python scripts/package_kaggle_model.py   --checkpoint-dir outputs/checkpoints   --output-zip outputs/kaggle_meter_model.zip   --arch-type s   --upload   --kaggle-model username/meter-nyu
```